In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import joblib


In [ ]:
df = pd.read_csv("smart_manufacturing_predictive_maintenance_oee.csv")

In [ ]:
print(df.head())
print(df.columns)
print(df.shape)
print(df.info())
print(df.describe())

In [ ]:
df["Failure_Next_Hour"].value_counts()

In [ ]:
sns.countplot(data=df, x="Failure_Next_Hour")
plt.title("Next-Hour Machine Failure Distribution")
plt.xlabel("Failure Next Hour")
plt.ylabel("Number of Records")
plt.show()

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
# Fill the only missing categorical values
df["Maintenance_Type"] = df["Maintenance_Type"].fillna("None")

In [ ]:
print("Duplicates:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("After removing duplicates:", df.shape)

In [ ]:
# Timestamp feature engineering
df["Timestamp"] = pd.to_datetime(df["Timestamp"])

df["Hour"] = df["Timestamp"].dt.hour
df["Day_of_Week"] = df["Timestamp"].dt.dayofweek
df["Month"] = df["Timestamp"].dt.month
df["Is_Weekend"] = (df["Timestamp"].dt.dayofweek >= 5).astype(int)

df = df.drop(columns=["Timestamp"])

In [ ]:
# Remove future information when predicting the next hour
df = df.drop(columns=["Energy_Next_Hour_kWh", "OEE_Next_Hour_pct"])

In [ ]:
# Separate X and y
X = df.drop("Failure_Next_Hour", axis=1)
y = df["Failure_Next_Hour"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
categorical_cols = X.select_dtypes(include=["object"]).columns

X_train = pd.get_dummies(
    X_train,
    columns=categorical_cols,
    drop_first=True
)

X_test = pd.get_dummies(
    X_test,
    columns=categorical_cols,
    drop_first=True
)

# Make training and testing columns identical
X_train, X_test = X_train.align(
    X_test,
    join="left",
    axis=1,
    fill_value=0
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

In [ ]:
# Create Random Forest model
model = RandomForestClassifier(
    n_estimators=50,
    max_depth=15,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X_train, y_train)

print("Model trained successfully!")

In [ ]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
# Save model and feature names for Streamlit
feature_names = X_train.columns.tolist()

joblib.dump(
    model,
    "predictive_maintenance_model.pkl",
    compress=3
)

joblib.dump(
    feature_names,
    "feature_names.pkl",
    compress=3
)

print("Model and feature names saved successfully!")

In [ ]:
# Feature importance
importance = pd.Series(
    model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False).head(15)

importance.plot(kind="bar", figsize=(10, 5))
plt.title("Top 15 Feature Importances")
plt.xlabel("Features")
plt.ylabel("Importance")
plt.show()

In [ ]:
# OEE analysis
print("Average OEE:", df["OEE_pct"].mean())
print("Average Availability:", df["Availability_pct"].mean())
print("Average Performance:", df["Performance_pct"].mean())
print("Average Quality:", df["Quality_pct"].mean())

In [ ]:
# OEE by machine
machine_oee = df.groupby("Machine_ID")["OEE_pct"].mean().sort_values(ascending=False)

machine_oee.plot(kind="bar", figsize=(10, 5))
plt.title("Average OEE by Machine")
plt.xlabel("Machine ID")
plt.ylabel("OEE %")
plt.show()